# Análise de Desempenho — Simulador de Livro de Ofertas

Este notebook apresenta a análise do simulador de livro de ofertas desenvolvido na disciplina de Estrutura de Dados em Python.

O sistema utiliza estruturas lineares implementadas manualmente por meio de nós encadeados, conforme solicitado no enunciado do trabalho.

A proposta envolve o uso de fila, pilha, lista encadeada ordenada e motor de casamento de ordens.

In [9]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

from ordem import OrdemNode
from fila import Fila
from pilha import Pilha
from lista_encadeada import ListaDuplamenteEncadeada
from motor_match import MotorMatch

## Estruturas Utilizadas

O simulador de livro de ofertas foi desenvolvido com estruturas de dados lineares implementadas manualmente por meio de nós encadeados.

As principais estruturas utilizadas no sistema são:

- **Ordem:** representa uma oferta de compra ou venda, contendo ID, tipo, preço, quantidade e timestamp.
- **Fila de Entrada:** armazena as ordens na sequência em que chegam ao sistema, seguindo o princípio FIFO.
- **Livro de Ofertas:** utiliza listas encadeadas ordenadas para organizar ordens de compra e venda por prioridade de preço.
- **Pilha:** registra os IDs das ordens inseridas com sucesso, permitindo a operação de desfazer.
- **Motor de Match:** realiza o casamento entre ordens de compra e venda quando os preços são compatíveis.

## Representação das Ordens

Cada ordem representa uma intenção de compra ou venda de um ativo financeiro.

A classe `OrdemNode` armazena as informações necessárias para o processamento das negociações:

- ID único;
- tipo da ordem (Compra ou Venda);
- preço;
- quantidade;
- timestamp.

In [2]:
ordem1 = OrdemNode("C", 10.50, 100)
ordem2 = OrdemNode("V", 11.00, 50)

print("Ordem 1")
print("ID:", ordem1.id)
print("Tipo:", ordem1.tipo)
print("Preço:", ordem1.preco)
print("Quantidade:", ordem1.quantidade)
print("Timestamp:", ordem1.timestamp)

print("\nOrdem 2")
print("ID:", ordem2.id)
print("Tipo:", ordem2.tipo)
print("Preço:", ordem2.preco)
print("Quantidade:", ordem2.quantidade)
print("Timestamp:", ordem2.timestamp)

Ordem 1
ID: 1
Tipo: C
Preço: 10.5
Quantidade: 100
Timestamp: 2026-06-23 22:37:26.706477

Ordem 2
ID: 2
Tipo: V
Preço: 11.0
Quantidade: 50
Timestamp: 2026-06-23 22:37:26.708645


### Análise dos Resultados

Observa-se que cada ordem recebe automaticamente um identificador único (ID) e um timestamp no momento da criação.

Essas informações permitem o controle individual das ordens e o registro da sequência temporal de entrada no sistema.

A criação de uma ordem possui complexidade O(1), pois apenas realiza atribuições de valores aos atributos do objeto.

## Fila de Entrada

A fila de entrada é responsável por armazenar as ordens antes do processamento pelo motor de negociação.

Essa estrutura segue o princípio FIFO (*First In, First Out*), ou seja, a primeira ordem inserida é a primeira ordem removida.

No contexto do simulador, isso garante que as ordens sejam processadas na mesma sequência em que chegaram ao sistema.

In [4]:
fila = Fila()

print("Fila vazia no início?", fila.is_empty())

fila.enqueue(ordem1)
fila.enqueue(ordem2)

print("Fila vazia após inserir ordens?", fila.is_empty())

primeira_ordem = fila.dequeue()
segunda_ordem = fila.dequeue()

print("\nPrimeira ordem removida da fila")
print("ID:", primeira_ordem.id)
print("Tipo:", primeira_ordem.tipo)

print("\nSegunda ordem removida da fila")
print("ID:", segunda_ordem.id)
print("Tipo:", segunda_ordem.tipo)

print("\nFila vazia no final?", fila.is_empty())

Fila vazia no início? True
Fila vazia após inserir ordens? False

Primeira ordem removida da fila
ID: 1
Tipo: C

Segunda ordem removida da fila
ID: 2
Tipo: V

Fila vazia no final? True


### Análise dos Resultados

A fila iniciou vazia, recebeu duas ordens e depois removeu essas ordens na mesma sequência em que foram inseridas.

A ordem de ID 1 foi removida antes da ordem de ID 2, demonstrando o comportamento FIFO da estrutura.

As operações de inserção (`enqueue`) e remoção (`dequeue`) possuem complexidade O(1), pois a fila mantém ponteiros para o início e para o fim da estrutura, evitando a necessidade de percorrer todos os nós.

## Sistema de Undo com Pilha

A pilha é utilizada para registrar os IDs das ordens inseridas com sucesso no livro de ofertas.

Essa estrutura segue o princípio LIFO (*Last In, First Out*), ou seja, o último elemento inserido é o primeiro a ser removido.

No contexto do simulador, isso permite identificar rapidamente a última ordem inserida, possibilitando a operação de desfazer.

In [5]:
pilha = Pilha()

print("Pilha vazia no início?", pilha.is_empty())

pilha.push(ordem1.id)
pilha.push(ordem2.id)

print("Pilha vazia após inserir IDs?", pilha.is_empty())

ultimo_id = pilha.pop()
proximo_id = pilha.pop()

print("\nÚltimo ID removido da pilha:", ultimo_id)
print("Próximo ID removido da pilha:", proximo_id)

print("\nPilha vazia no final?", pilha.is_empty())

Pilha vazia no início? True
Pilha vazia após inserir IDs? False

Último ID removido da pilha: 2
Próximo ID removido da pilha: 1

Pilha vazia no final? True


### Análise dos Resultados

A pilha iniciou vazia, recebeu os IDs das duas ordens criadas e removeu esses IDs na ordem inversa de inserção.

O ID 2 foi removido antes do ID 1, demonstrando o comportamento LIFO da estrutura.

As operações `push` e `pop` possuem complexidade O(1), pois a pilha realiza inserções e remoções diretamente no topo, sem necessidade de percorrer todos os nós.

## Livro de Ofertas

O livro de ofertas é responsável por armazenar as ordens de compra e venda que aguardam negociação.

Para atender aos requisitos do simulador, foi utilizada uma lista duplamente encadeada ordenada.

Essa estrutura permite manter as ordens organizadas por preço, facilitando a identificação das melhores ofertas disponíveis para compra e venda.

No sistema, são utilizadas duas listas:

- Lista de Compras: mantida em ordem decrescente de preço.
- Lista de Vendas: mantida em ordem crescente de preço.

### Organização das Ordens

#### Lista de Compras

```text
25 → 20 → 18 → 15
```

A melhor oferta de compra fica no início da lista.

#### Lista de Vendas

```text
10 → 12 → 14 → 18
```

A melhor oferta de venda também fica no início da lista.

In [7]:
compras = ListaDuplamenteEncadeada(ordem_crescente=False)

compra1 = OrdemNode("C", 20.0, 100)
compra2 = OrdemNode("C", 15.0, 50)
compra3 = OrdemNode("C", 25.0, 80)

compras.inserir_ordenado(compra1)
compras.inserir_ordenado(compra2)
compras.inserir_ordenado(compra3)

print("Livro de Compras:")
compras.exibir()

Livro de Compras:
ID: 5 | Preço: 25.0 | Quantidade: 80
ID: 3 | Preço: 20.0 | Quantidade: 100
ID: 4 | Preço: 15.0 | Quantidade: 50


In [8]:
vendas = ListaDuplamenteEncadeada(ordem_crescente=True)

venda1 = OrdemNode("V", 18.0, 100)
venda2 = OrdemNode("V", 10.0, 50)
venda3 = OrdemNode("V", 14.0, 80)

vendas.inserir_ordenado(venda1)
vendas.inserir_ordenado(venda2)
vendas.inserir_ordenado(venda3)

print("Livro de Vendas:")
vendas.exibir()

Livro de Vendas:
ID: 7 | Preço: 10.0 | Quantidade: 50
ID: 8 | Preço: 14.0 | Quantidade: 80
ID: 6 | Preço: 18.0 | Quantidade: 100


### Análise dos Resultados

Observa-se que as ordens são inseridas automaticamente na posição correta da lista, mantendo a ordenação por preço.

Na lista de compras, as maiores ofertas permanecem no início da estrutura. Já na lista de vendas, as menores ofertas ocupam as primeiras posições.

Esse comportamento é fundamental para o funcionamento do livro de ofertas, pois permite identificar rapidamente as melhores oportunidades de negociação.

A operação de inserção ordenada possui complexidade O(n), pois pode ser necessário percorrer todos os nós da lista até encontrar a posição correta para a nova ordem.

### Importância da Ordenação

A manutenção automática da ordenação permite que as melhores ofertas fiquem sempre posicionadas no início das listas.

Dessa forma, o sistema pode acessar rapidamente a melhor ordem de compra e a melhor ordem de venda por meio da cabeça da lista, sem necessidade de percorrer toda a estrutura.

Essa característica é fundamental para o funcionamento eficiente do motor de casamento de ordens.

## Motor de Match

O motor de match é responsável por verificar se existe compatibilidade entre as melhores ordens de compra e venda.

Como as listas estão ordenadas, o motor compara apenas o topo da lista de compras com o topo da lista de vendas.

A regra principal é:

```text
Se preço da compra >= preço da venda,
então ocorre uma transação.

In [11]:
livro_compras = ListaDuplamenteEncadeada(ordem_crescente=False)
livro_vendas = ListaDuplamenteEncadeada(ordem_crescente=True)

ordem_compra = OrdemNode("C", 20.0, 100)
ordem_venda = OrdemNode("V", 15.0, 50)

livro_compras.inserir_ordenado(ordem_compra)
livro_vendas.inserir_ordenado(ordem_venda)

motor = MotorMatch()

transacoes = motor.executar(livro_compras, livro_vendas)

print("Transações realizadas:")
for transacao in transacoes:
    print(transacao)

print("\nLivro de Compras após o match:")
livro_compras.exibir()

print("\nLivro de Vendas após o match:")
if livro_vendas.is_empty():
    print("Livro vazio")
else:
    livro_vendas.exibir()

Transações realizadas:
Compra #11 x Venda #12 | 50 ações a R$20.00

Livro de Compras após o match:
ID: 11 | Preço: 20.0 | Quantidade: 50

Livro de Vendas após o match:
Livro vazio


### Análise dos Resultados

Neste exemplo, a ordem de compra possui preço maior que a ordem de venda. Portanto, a condição para o casamento foi satisfeita.

Como a compra aceitava pagar R$ 20,00 e a venda aceitava receber R$ 15,00, o motor executou uma transação entre as duas ordens.

A quantidade negociada foi definida pelo menor volume disponível entre compra e venda. Como a venda possuía quantidade 50 e a compra possuía quantidade 100, foram negociadas 50 ações.

O preço de execução foi R$ 20,00, pois o motor utiliza o preço da ordem que chegou primeiro ao sistema. Neste caso, a ordem de compra foi criada antes da ordem de venda.

Após a transação, a ordem de venda foi totalmente atendida e removida do livro. A ordem de compra permaneceu com quantidade restante de 50 ações. Por isso, o livro de vendas aparece como vazio após o match.